# 05 · Bonds: Fixed, Floating, and Hybrids

Finstack ships multiple bond constructors: Treasury-style helpers, fully parameterized builders, floating-rate notes, amortizing loans, and schedule-driven hybrids. This notebook highlights the patterns used in exotic credit and private credit desks.

### Learning Objectives
- Build liquid discount/forward markets once and reuse them across bond variants
- Instantiate fixed, floating, zero-coupon, and Treasury bonds with helper constructors
- Feed custom cashflow schedules (PIK/amortizing) into `Bond.from_cashflows`
- Compute asset-swap spreads and analytics such as clean price, yield, Z-spread, and DV01

In [1]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, ForwardCurve
from finstack.valuations.cashflow import (
    CashFlowBuilder,
    ScheduleParams,
    FixedCouponSpec,
    CouponType,
    AmortizationSpec,
)
from finstack.valuations.instruments import Bond
from finstack.valuations.metrics import MetricId
from finstack.valuations.pricer import standard_registry

as_of = date(2025, 1, 2)

def build_market(as_of: date) -> MarketContext:
    market = MarketContext()
    usd_ois = DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9975), (1.0, 0.9940), (3.0, 0.9725), (5.0, 0.9440)],
    )
    treasury = DiscountCurve(
        "USD-TREASURY",
        as_of,
        [(0.0, 1.0), (5.0, 0.9750), (10.0, 0.9300)],
    )
    sofr_3m = ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.045), (1.0, 0.0465), (3.0, 0.0480), (5.0, 0.0495)],
        base_date=as_of,
    )
    market.insert_discount(usd_ois)
    market.insert_discount(treasury)
    market.insert_forward(sofr_3m)
    return market

market = build_market(as_of)
registry = standard_registry()
notional = Money(10_000_000, USD)


## 1. Core Constructors
The bindings expose ergonomic helpers for common structures. We value a Treasury-style bond, a standard corporate fixed-rate note, a floating-rate note, and a zero-coupon bond.

In [2]:
treasury = (
    Bond.builder("UST-5Y")
    .money(notional)
    .coupon_rate(0.03625)
    .issue(date(2023, 11, 15))
    .maturity(date(2028, 11, 15))
    .disc_id("USD-TREASURY")
    .frequency("semiannual")
    .build()
)
corp_fixed = (
    Bond.builder("ACME-31")
    .money(notional)
    .coupon_rate(0.045)
    .issue(as_of)
    .maturity(date(2030, 1, 2))
    .disc_id("USD-OIS")
    .frequency("semiannual")
    .build()
)
frn = (
    Bond.builder("ACME-FRN")
    .money(notional)
    .coupon_rate(0.0)
    .issue(as_of)
    .maturity(date(2029, 1, 2))
    .disc_id("USD-OIS")
    .forward_curve("USD-SOFR-3M")
    .float_margin_bp(125.0)
    .frequency("quarterly")
    .build()
)
zcb = (
    Bond.builder("ACME-ZCB")
    .money(notional)
    .coupon_rate(0.0)
    .issue(as_of)
    .maturity(date(2030, 1, 2))
    .disc_id("USD-OIS")
    .frequency("annual")
    .build()
)

for instrument in (treasury, corp_fixed, frn, zcb):
    res = registry.price(instrument, "discounting", market, as_of=as_of)
    print(f"{instrument.instrument_id:<12} PV = {res.value.amount:,.2f} {res.value.currency}")


UST-5Y       PV = 11,237,567.25 USD
ACME-31      PV = 11,618,469.08 USD
ACME-FRN     PV = 11,913,071.05 USD
ACME-ZCB     PV = 9,429,866.08 USD


## 2. Cashflow Builder Integration
For bespoke structures (PIK toggles, amortization, callable schedules) create a schedule via `CashFlowBuilder` and wrap it with `Bond.from_cashflows`.

In [3]:
builder = CashFlowBuilder.new()
builder.principal(
    amount=notional.amount,
    currency=USD,
    issue=as_of,
    maturity=date(2032, 1, 2),
)
# Semi-annual 5% coupon with 60% cash / 40% PIK split
builder.fixed_cf(
    FixedCouponSpec.new(
        rate=0.05,
        schedule=ScheduleParams.semiannual_30360(),
        coupon_type=CouponType.split(0.60, 0.40),
    )
)
# Linear amortization to 70% of original notional
builder.amortization(AmortizationSpec.linear_to(Money(7_000_000, USD)))
custom_schedule = builder.build_with_curves(None)
print("First five flows (date, amount, kind):")
for flow in custom_schedule.flows()[:5]:
    dt, amt, kind, accr, reset = flow.to_tuple()
    print(f"  {dt} {str(kind):<10} {amt.amount:,.2f} {amt.currency}")

custom_bond = (
    Bond.builder("ACME-CUSTOM")
    .cashflows(custom_schedule)
    .disc_id("USD-OIS")
    .quoted_clean_price(99.40)
    .build()
)
print("Custom schedule PV:", registry.price(custom_bond, "discounting", market, as_of=as_of).value)


First five flows (date, amount, kind):
  2025-01-02 notional   -10,000,000.00 USD
  2025-07-02 stub       150,000.00 USD
  2025-07-02 amortization 214,285.71 USD
  2025-07-02 pik        100,000.00 USD
  2026-01-02 fixed      148,285.71 USD
Custom schedule PV: USD 12325669.96


## 3. Asset-Swap Spreads
Given a floating reference curve and quoted dirty price, the registry can compute par and market asset-swap spreads. This is useful for relative-value screens.

In [4]:
market_dirty_price = 0.90 * corp_fixed.notional.amount
par_asw, mkt_asw = registry.asw_forward(
    corp_fixed,
    market,
    "USD-SOFR-3M",
    90.0,
    market_dirty_price,
)
print(f"Par ASW:  {par_asw:.2f} bp")
print(f"Mkt ASW: {mkt_asw:.2f} bp")


Par ASW:  -0.01 bp
Mkt ASW: -0.03 bp


## 4. Bond Analytics
Combine helper metrics to capture pricing, yield, spread, and duration packages in a single call.

In [5]:
metrics = [
    "clean_price",
    "dirty_price",
    "accrued",
    "ytm",
    "duration_mod",
    "duration_mac",
    "dv01",
    "convexity",
    "z_spread",
    "i_spread",
]
res = registry.price_with_metrics(
    corp_fixed,
    "discounting",
    market,
    as_of=as_of,
    metrics=metrics,
)
for name in metrics:
    print(f"{name:>12}: {res.measures.get(name)}")


 clean_price: 11618469.075430868
 dirty_price: 11618469.075430868
     accrued: 0.0
         ytm: 0.011589979902105032
duration_mod: 4.547024017118457
duration_mac: 4.573373975604853
        dv01: -5387.273167870939
   convexity: 24.09720925986837
    z_spread: -2.0580205511060233e-11
    i_spread: -0.00013257893161788999


## Summary
- Helper constructors (`treasury`, `fixed_semiannual`, `floating`, `zero_coupon`) cover the most common issuance styles.
- `CashFlowBuilder` unlocks bespoke amortizing/PIK programs that can be promoted into bonds for pricing and risk.
- Asset-swap utilities and metric bundles surface trader-ready analytics (clean price, yield, spreads, DV01) without hand-written ladders.